# 01 - Dataset Preparation

Builds the VQAv2 subset used by every other notebook.

**Pipeline :** download -> extract -> sample images -> join questions with annotations -> write one CSV per split -> build the answer space.

Sampling plan is driven entirely by `vqa/config.py` :
* `TOTAL_IMAGES` (default 5000) with a 73 / 13.5 / 13.5 train-val-test ratio
* train images come from `train2014`, val and test from **disjoint slices** of `val2014`

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from vqa import config as C
C.checkEnv()
C.makeDirs()
C.showConfig()

### Download and extract

Around 13 GB for `train2014` + 6 GB for `val2014`, so this cell is the slow one. Everything is skipped if it already exists on disk, so the notebook is safe to rerun.

In [ ]:
from vqa import dataPrep as P

#----------------------------------------------------------------------------------------<Questions + Annotations>--------||
for key in ["trainQstn", "valQstn", "trainAnnot", "valAnnot"]:
    P.fetchZip(key)

In [ ]:
#----------------------------------------------------------------------------------------<Images>-------------------------||
# The big ones. Comment out whichever is already unpacked locally.
P.fetchZip("trainImg")
P.fetchZip("valImg")

### Build the splits

Samples, joins each question to its annotation, writes the CSV and copy these into `Dataset/images/`.

In [ ]:
trainFrame = P.buildSplit("train")

In [ ]:
valFrame = P.buildSplit("val")

In [ ]:
testFrame = P.buildSplit("test")

In [ ]:
trainFrame.head()

### Sanity checks

In [ ]:
frames = P.checkSplits()

### Answer space

Built from **train only**, top-K most frequent, sorted for determinism, with `<unk>` pinned at index 0.

In [ ]:
from vqa import dataProc as D

answerSpace = D.buildAnswerSpace()
answerSpace[:10]

In [ ]:
#----------------------------------------------------------------------------------------<Distribution check>------------||
import matplotlib.pyplot as plt
from collections import Counter

counts = Counter(trainFrame["answer_val"].astype(str))
top30  = counts.most_common(30)

plt.figure(figsize=(14, 4))
plt.bar([a for a, _ in top30], [c for _, c in top30])
plt.xticks(rotation=75, ha="right")
plt.title("30 most frequent answers - train split")
plt.tight_layout()
plt.savefig(C.FIGURE_DIR / "answerDistribution.png", dpi=150)
plt.show()

In [ ]:
#----------------------------------------------------------------------------------------<Eyeball a sample>--------------||
import json
from PIL import Image
from IPython.display import display

row = trainFrame.sample(1, random_state=C.SEED).iloc[0]
display(Image.open(C.IMAGE_DIR / row["image_name"]))

print("Question       :", row["question_val"])
print("Consensus      :", row["answer_val"])
print("All 10 answers :", json.loads(row["answers"]))
print("Answer type    :", row["answer_type"])

### Pre resize the images

The collator decodes a full size COCO JPEG and resizes it to 224 for every image, every epoch. That is 15-25 ms of CPU per image, which starves the GPU. Doing it once here cuts per batch data prep by roughly 5-10x.

ViT resizes to a square 224x224 anyway, so this changes nothing about what the model sees. Originals stay untouched in `Dataset/images/`; set `C.IMAGE_SIZE_DIR = None` to go back to them.

In [ ]:
resized = P.resizeImages(size=224)
print("collator will read from :", C.activeImageDir())

### Disk cleanup

`Dataset/images/` now holds every image the project needs. The 19 GB of COCO zips and unpacked folders are dead weight from here on.

`cleanupRaw` refuses to run unless all three CSVs exist and every image they reference is present, and it is a dry run by default. Only pass `confirm=True` once `checkSplits()` above came back clean, because undoing this means downloading 19 GB again.

In [ ]:
P.diskUsage()

In [ ]:
#----------------------------------------------------------------------------------------<Dry run first>-----------------||
P.cleanupRaw()

In [ ]:
#----------------------------------------------------------------------------------------<For real>-----------------------||
# Uncomment once the checks above are clean.
# P.cleanupRaw(confirm=True)
# P.diskUsage()